In [1]:
!pip install ipykernel torch transformers datasets accelerate evaluate scikit-learn


   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [evaluate]
   ---------------------------------------- 2/2 [evaluate]




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import torch

# GPU 인식 확인
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"▶ 현재 사용 중인 장치: {device.upper()}")

# 가상환경 충돌 방지용 설정
os.environ["TOKENIZERS_PARALLELISM"] = "false"

▶ 현재 사용 중인 장치: CPU


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 데이터 다운로드
dataset = load_dataset("klue", "ynat")

# 2. 모델 및 토크나이저 다운로드 (뉴스 카테고리가 7개이므로 num_labels=7)
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

print(f"✔ 데이터셋 로드 완료 (학습 데이터: {len(dataset['train'])}개)")

▶ 데이터셋 다운로드 중...


README.md: 0.00B [00:00, ?B/s]

c:\Users\hi\miniconda3\envs\langchain_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hi\.cache\huggingface\hub\datasets--klue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For 

train-00000-of-00001.parquet:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

c:\Users\hi\miniconda3\envs\langchain_env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✔ 데이터셋 로드 완료 (학습 데이터: 45678개)


In [4]:
def preprocess_function(examples):
    # 뉴스 타이틀 최대 길이를 64로 제한하고 부족하면 패딩을 채웁니다
    return tokenizer(examples["title"], truncation=True, max_length=64, padding="max_length")

print("▶ 전체 텍스트 데이터 토크나이징 변환 중...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)
print("✔ 전처리 완료!")


▶ 전체 텍스트 데이터 토크나이징 변환 중...


Map:   0%|          | 0/45678 [00:00<?, ? examples/s]

Map:   0%|          | 0/9107 [00:00<?, ? examples/s]

✔ 전처리 완료!


In [5]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

# 정확도(Accuracy)와 다중분류용 F1-score 평가지표 정의
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

# 학습 환경 설정 (GPU 메모리에 맞춰 batch_size를 16으로 안전하게 설정)
training_args = TrainingArguments(
    output_dir="./bert_ynat_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),  # GPU 사용 시 연산 가속 적용
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("▶ BERT 미세조정 학습을 시작합니다. (시간이 다소 소요될 수 있습니다)")
trainer.train()

# 제일 우수한 성능을 낸 체크포인트를 별도 폴더에 저장
trainer.save_model("./best_bert_ynat_model")
tokenizer.save_pretrained("./best_bert_ynat_model")
print("✔ 학습 완료 및 최적 모델 저장 성공!")


▶ BERT 미세조정 학습을 시작합니다. (시간이 다소 소요될 수 있습니다)


  0%|          | 0/8565 [00:00<?, ?it/s]

c:\Users\hi\miniconda3\envs\langchain_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 1.0566, 'grad_norm': 12.18890380859375, 'learning_rate': 1.9766491535318158e-05, 'epoch': 0.04}
{'loss': 0.4665, 'grad_norm': 11.502086639404297, 'learning_rate': 1.9532983070636315e-05, 'epoch': 0.07}
{'loss': 0.4391, 'grad_norm': 20.928508758544922, 'learning_rate': 1.9299474605954468e-05, 'epoch': 0.11}
{'loss': 0.4336, 'grad_norm': 4.375838279724121, 'learning_rate': 1.9065966141272624e-05, 'epoch': 0.14}
{'loss': 0.4162, 'grad_norm': 9.31811809539795, 'learning_rate': 1.8832457676590777e-05, 'epoch': 0.18}
{'loss': 0.4102, 'grad_norm': 9.5929594039917, 'learning_rate': 1.8598949211908934e-05, 'epoch': 0.21}
{'loss': 0.4098, 'grad_norm': 7.025069713592529, 'learning_rate': 1.8365440747227087e-05, 'epoch': 0.25}
{'loss': 0.392, 'grad_norm': 9.033075332641602, 'learning_rate': 1.8131932282545244e-05, 'epoch': 0.28}
{'loss': 0.4318, 'grad_norm': 12.746681213378906, 'learning_rate': 1.78984238178634e-05, 'epoch': 0.32}
{'loss': 0.4275, 'grad_norm': 5.964284896850586, 'learning

  0%|          | 0/570 [00:00<?, ?it/s]

{'eval_loss': 0.38229233026504517, 'eval_accuracy': 0.8649390578675744, 'eval_f1_macro': 0.8685767260513966, 'eval_runtime': 326.6494, 'eval_samples_per_second': 27.88, 'eval_steps_per_second': 1.745, 'epoch': 1.0}


c:\Users\hi\miniconda3\envs\langchain_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.3065, 'grad_norm': 2.1544084548950195, 'learning_rate': 1.3228254524226505e-05, 'epoch': 1.02}
{'loss': 0.276, 'grad_norm': 10.267295837402344, 'learning_rate': 1.2994746059544661e-05, 'epoch': 1.05}
{'loss': 0.272, 'grad_norm': 7.831373691558838, 'learning_rate': 1.2761237594862815e-05, 'epoch': 1.09}
{'loss': 0.2744, 'grad_norm': 10.644021034240723, 'learning_rate': 1.252772913018097e-05, 'epoch': 1.12}


In [ ]:
from transformers import pipeline

# 저장된 로컬 모델을 불러와 추론 파이프라인 생성
classifier = pipeline(
    "text-classification", 
    model="./best_bert_ynat_model", 
    tokenizer="./best_bert_ynat_model"
)

# 숫자로 반환되는 결과를 직관적인 텍스트로 치환하기 위한 사전
labels_map = {
    "LABEL_0": "IT과학", "LABEL_1": "경제", "LABEL_2": "사회", 
    "LABEL_3": "생활문화", "LABEL_4": "세계", "LABEL_5": "스포츠", "LABEL_6": "정치"
}

# 테스트용 문장 리스트
custom_news = [
    "토트넘 손흥민, 경기 종료 직전 극적 결승골 폭발",
    "미국 연준, 예상 깨고 오늘 새벽 기준 금리 깜짝 인상 발표",
    "국내 연구진, 세계 최초로 실온 초전도체 물질 개발 성공 주장"
]

print("\n==== [ 나만의 BERT 모델 뉴스 예측 결과 ] ====")
for title in custom_news:
    pred = classifier(title)[0]
    mapped_label = labels_map[pred['label']]
    print(f"📰 입력 제목 : {title}")
    print(f"🎯 예측 분류 : {mapped_label} (확률: {pred['score']*100:.2f}%)")
    print("-" * 50)
